# Amazon ML Challenge 2026 — Kaggle run
CPU-only (no GPU needed until optional reranker). Attach your dataset first: Settings > Data > Add Input > your private 1GB dataset.
Expected input path: `/kaggle/input/<your-dataset>/...` containing `train/`, `test/`. Adjust `DATA_DIR` below.

In [1]:
!pip install -q lightgbm rapidfuzz
import os
DATA_DIR = '/kaggle/input/datasets/ayushastiker/amazon-ml-challenge/student_resource/dataset'  # <-- EDIT THIS
SRC = '/kaggle/working/code_ber/business_entity_resolution/src'  # upload repo files or git clone your repo here
OUT = '/kaggle/working/output'
os.makedirs(OUT, exist_ok=True)
print(os.listdir(DATA_DIR))
print(os.listdir(os.path.join(DATA_DIR,'train'))[:10])

['test', 'train']
['train_ground_truth.tsv', 'train_source3.tsv', 'train_source2.tsv', 'train_source1.tsv']


In [2]:
# Phase 1 — audit (CPU, ~2-5 min). Uses your real numbers: S1 2.2M, singleton 5.6%.
!python $SRC/audit.py --data-dir $DATA_DIR

=== Sizes ===
S1: 2206821  S2: 5034616  S3: 5285603  GT rows: 2206821

=== Country distribution (train) ===
S1 {'US': 1323633, 'India': 883188}
S2 {'India': 2017799, 'US': 3016817}
S3 {'US': 3170056, 'India': 2115547}

=== Missingness (train) ===
S2.business_address: 168967 empty (3.356%)
S3.business_address: 175916 empty (3.328%)

=== Ground-truth degree distribution ===
  degree=0: 123247 entities (5.585%)
  degree=1: 119157 entities (5.399%)
  degree=2: 375212 entities (17.002%)
  degree=3: 530841 entities (24.055%)
  degree=4: 484115 entities (21.937%)
  degree=5: 321957 entities (14.589%)
  degree=6: 164868 entities (7.471%)
  degree=7: 63968 entities (2.899%)
  degree=8: 18680 entities (0.846%)
  degree=9: 4205 entities (0.191%)
  degree=10: 534 entities (0.024%)
  degree=11: 37 entities (0.002%)
Singleton rate: 0.0558
S2-only matches: 143029  S3-only matches: 164498  both: 1776047

S2/S3 records matched to >1 Source-1 entity: 0 
If this is non-zero, do NOT assume a global one-to

In [3]:
# Phase 2 — blocking recall on sample (CPU). Tune max-df here before full run.
!python $SRC/pipeline.py --help
!python $SRC/bench_blocking.py --data-dir $DATA_DIR --n 5000 --max-df 300

usage: pipeline.py [-h] --data-dir DATA_DIR --out-dir OUT_DIR
                   [--n-splits N_SPLITS] [--seed SEED] [--sample-s1 SAMPLE_S1]
                   [--max-df MAX_DF] [--prefix-len PREFIX_LEN]
                   [--max-pairs-per-prefix-key MAX_PAIRS_PER_PREFIX_KEY]
                   [--neg-per-pos-cap NEG_PER_POS_CAP] [--use-tfidf]
                   [--validate]

options:
  -h, --help            show this help message and exit
  --data-dir DATA_DIR   dir containing train/ and test/ subfolders
  --out-dir OUT_DIR     dir to write matching_results.tsv /
                        candidate_pairs.tsv
  --n-splits N_SPLITS
  --seed SEED
  --sample-s1 SAMPLE_S1
  --max-df MAX_DF
  --prefix-len PREFIX_LEN
  --max-pairs-per-prefix-key MAX_PAIRS_PER_PREFIX_KEY
  --neg-per-pos-cap NEG_PER_POS_CAP
  --use-tfidf           small-sample TF-IDF blocking only
  --validate            also run utils/validate_submission.py if found
                        alongside --data-dir
load 173.3s S1=22

In [4]:
# Phase 3 — sampled train: matcher + calibration + threshold (CPU). No test inference.
!python $SRC/bench_train.py --data-dir $DATA_DIR --n 5000 --max-df 300 --n-splits 3

load 169.3s
pairs=2,653,273 pos=17,162 recall=0.9920
train pairs=272,145 pos_rate=0.0631
OOF macro F0.5=0.9840 @t=0.625 | singleton-baseline=0.0554


In [ ]:
# Phase 4 — full run (CPU, long: hours). Run once recall >= 0.90 and sampled OOF looks sane.
# Start with --sample-s1 50000 before attempting full.
!python $SRC/pipeline.py --data-dir $DATA_DIR --out-dir $OUT --n-splits 3 --sample-s1 20000 --max-df 100 --neg-per-pos-cap 15 --validate

[11:33:39] Loading training sources...
[11:36:41] Train sizes: S1=2206821 S2=5034616 S3=5285603 | GT rows=2206821 | singleton rate=0.056
[11:36:41] --sample-s1: subsampling to 20000 S1 (matched S2/S3 kept, distractors proportional).
[11:36:46] Sampled train: S1=20000 S2=93031 S3=95120
[11:36:46] Loading test sources...
[11:37:52] Test sizes: S1=1732544 S2=4887273 S3=5082316 | test countries=['France', 'India', 'US']
[11:37:52] Normalizing text fields...
[11:44:53] Blocking train S1 vs S2 (max_df=100)...
[11:45:01] Blocking train S1 vs S3 (max_df=100)...
[11:45:11] Train candidate pairs: 5,482,554
[11:45:11] Train candidate recall: 0.9773
[11:45:11] Building pairwise feature frame (vectorized merges)...


In [ ]:
# Phase 5 — validate + inspect outputs (always before submitting)
!ls -lh $OUT
!head -5 $OUT/matching_results.tsv
!python utils/validate_submission.py --matching $OUT/matching_results.tsv --candidate $OUT/candidate_pairs.tsv --test-dir $DATA_DIR/test